#### Load Dataset

In [1]:
import pandas as pd
import ast

data_path = "../data/usc-x-24-us-election/part_1/"

files = [
    "may_july_chunk_1.csv.gz",
    "may_july_chunk_2.csv.gz",
    "may_july_chunk_3.csv.gz",
    "may_july_chunk_4.csv.gz",
    "may_july_chunk_5.csv.gz"
]

dfs = []
for f in files:
    temp = pd.read_csv(data_path + f, compression='gzip')
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df.shape

(250000, 32)

In [2]:
df_sample = df[['id', 'text']].dropna().sample(30, random_state=42)
df_sample = df_sample.reset_index(drop=True)
df_sample.shape

(30, 2)

In [3]:
gallup_issues_map = {
    "The economy": (
        "inflation is killing us, prices are too high, cost of living, grocery prices, gas prices, "
        "rent is too expensive, unemployment, jobs report, wages, economy is tanking, recession, "
        "stock market, interest rates, fed reserve, bidenomics, economic growth, GDP, shrinkflation, "
        "can't afford groceries, everything costs more, 5x what it was, printing money"
    ),
    "Democracy in the U.S.": (
        "election integrity, stolen election, voter fraud, rigged election, January 6th, "
        "threat to democracy, MAGA, Trump 2024, Biden 2024, presidential election, voting rights, "
        "ballot harvesting, mail-in ballots, deep state, political corruption, drain the swamp, "
        "Democrat agenda, Republican agenda, two-party system, political polarization, "
        "Trump should be president, Biden should resign, vote them out"
    ),
    "Terrorism and national security": (
        "domestic terrorist attack, ISIS, Al Qaeda, jihad, sleeper cells, "
        "terror on American soil, FBI terror watch list, radical extremists, "
        "homeland security threat — NOT Russia, NOT Ukraine, NOT China, NOT Israel"
    ),
    "Types of Supreme Court justices candidates would pick": (
        "Supreme Court nominee, SCOTUS pick, judicial appointment, originalist judge, "
        "liberal judge, conservative justice, lifetime appointment, Roe v Wade overturned, "
        "packing the court, court packing, judicial philosophy, constitutional interpretation, "
        "who will they nominate, next Supreme Court justice"
    ),
    "Immigration": (
        "open borders, illegal aliens, undocumented immigrants, border wall, border security, "
        "migrants crossing the border, asylum seekers, deportation, ICE, DHS, sanctuary cities, "
        "Biden opened the border, invasion at the border, Title 42, mass migration, "
        "illegal immigration crisis, border patrol, cartel, gotaways, fentanyl at the border"
    ),
    "Education": (
        "public schools, student loan forgiveness, college tuition, school debt, "
        "critical race theory in schools, parents rights in education, school board, "
        "teachers union, homeschooling, college affordability, school choice, vouchers, "
        "indoctrinating kids, curriculum, common core, DEI on campus"
    ),
    "Healthcare": (
        "healthcare costs, medical bills, health insurance, Obamacare, ACA, "
        "Medicare for all, Medicaid, prescription drug prices, insulin costs, "
        "can't afford healthcare, hospital bills, pre-existing conditions, "
        "universal healthcare, Big Pharma, drug companies, mental health crisis"
    ),
    "Gun policy": (
        "Second Amendment, gun rights, gun control, mass shooting, AR-15, "
        "assault weapons ban, background checks, red flag laws, NRA, "
        "right to bear arms, gun violence, school shooting, taking our guns, "
        "armed citizens, concealed carry, gun free zones don't work"
    ),
    "Abortion": (
        "abortion rights, pro-life, pro-choice, Roe v Wade, abortion ban, "
        "right to choose, killing babies, women's reproductive rights, "
        "abortion access, heartbeat bill, late term abortion, abortion clinic, "
        "planned parenthood, defund planned parenthood, life begins at conception"
    ),
    "Taxes": (
        "tax hikes, raising taxes, tax cuts, corporate tax, billionaire tax, "
        "IRS, paying too much in taxes, tax the rich, Trump tax cuts, "
        "death tax, estate tax, government taking my money, tax and spend, "
        "fair share in taxes, fiscal policy, tax burden"
    ),
    "Crime": (
        "crime is out of control, defund the police, back the blue, law and order, "
        "violent crime, murder rate, carjacking, shoplifting, soft on crime, "
        "criminal justice reform, prison reform, police brutality, "
        "George Soros DA, revolving door justice, catch and release, "
        "smash and grab, crime wave, fentanyl deaths"
    ),
    "Distribution of income and wealth in the U.S.": (
        "wealth inequality, the rich keep getting richer, income gap, "
        "billionaires vs working class, Wall Street greed, corporate greed, "
        "wealth gap, working class struggles, middle class is dying, "
        "1 percent owns everything, economic inequality, living paycheck to paycheck, "
        "CEOs making millions while workers suffer"
    ),
    "The federal budget deficit": (
        "national debt, deficit spending, government spending out of control, "
        "trillions in debt, wasteful spending, government waste, bloated budget, "
        "debt ceiling, fiscal responsibility, spending taxpayer money, "
        "printing money, borrowed from China, out of control spending, pork barrel"
    ),
    "Foreign affairs": (
        "U.S. foreign policy, America's role in the world, foreign aid, "
        "giving money to other countries, America first, globalism, "
        "United Nations, NATO allies, military intervention, regime change, "
        "why are we policing the world, sending billions overseas, diplomacy"
    ),
    "Situation in Middle East between Israelis and Palestinians": (
        "Israel Gaza war, Hamas October 7th attack, Palestinian civilians, "
        "IDF, Netanyahu, bombing Gaza, hostages in Gaza, ceasefire, "
        "free Palestine, pro-Israel, humanitarian crisis in Gaza "
        "— NOT Russia, NOT China, NOT Ukraine"
    ),
    "Energy policy": (
        "gas prices, oil prices, drill baby drill, keystone pipeline, "
        "green new deal, solar panels, wind energy, fossil fuels, "
        "energy independence, OPEC, electric vehicles, EV mandate, "
        "energy crisis, natural gas, fracking, carbon tax, energy costs"
    ),
    "Relations with Russia": (
        "Putin, Russian invasion of Ukraine, Zelensky, NATO vs Russia, "
        "sending weapons to Ukraine, funding Ukraine, Russian military aggression, "
        "nuclear threat from Russia, Kremlin, U.S. sanctions on Russia, "
        "war in Eastern Europe — NOT China, NOT Middle East, NOT Israel"
    ),
    "Race relations": (
        "systemic racism, white privilege, Black Lives Matter, BLM, "
        "racial inequality, racial justice, civil rights, police racism, "
        "critical race theory, reverse racism, affirmative action, "
        "racial divide, white supremacy, DEI, diversity equity inclusion, "
        "reparations, discrimination, racial profiling"
    ),
    "Relations with China": (
        "CCP, Chinese Communist Party, Taiwan invasion, TikTok ban, "
        "spy balloon, Chinese espionage, trade war with China, fentanyl from China, "
        "South China Sea, standing up to China, Xi Jinping "
        "— NOT Russia, NOT Ukraine, NOT Middle East"
    ),
    "Trade with other nations": (
        "tariffs, trade war, trade deficit, American jobs going overseas, "
        "outsourcing, bring jobs back to America, free trade, NAFTA, USMCA, "
        "imports and exports, trade deal, protecting American industry, "
        "buy American, manufacturing jobs, offshoring, trade imbalance"
    ),
    "Climate change": (
        "climate change is a hoax, global warming, carbon emissions, "
        "green new deal, renewable energy, climate crisis, net zero, "
        "Paris agreement, carbon footprint, electric vehicles, "
        "extreme weather, wildfires, sea level rising, climate activists, "
        "fossil fuels vs clean energy, ESG"
    ),
    "Transgender rights": (
        "transgender athletes, trans women in women's sports, gender identity, "
        "gender affirming care, puberty blockers, drag shows, "
        "protecting children from gender ideology, bathroom bills, "
        "nonbinary, pronouns, LGBTQ rights, trans rights are human rights, "
        "don't say gay, parental rights vs gender ideology"
    ),
}
gallup_issues = list(gallup_issues_map.keys())
gallup_descriptions = list(gallup_issues_map.values())

issues_text = "\n".join(gallup_issues)
print(issues_text)

The economy
Democracy in the U.S.
Terrorism and national security
Types of Supreme Court justices candidates would pick
Immigration
Education
Healthcare
Gun policy
Abortion
Taxes
Crime
Distribution of income and wealth in the U.S.
The federal budget deficit
Foreign affairs
Situation in Middle East between Israelis and Palestinians
Energy policy
Relations with Russia
Race relations
Relations with China
Trade with other nations
Climate change
Transgender rights


In [4]:
import ollama
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def embed(text):
    response = ollama.embed(
        model="nomic-embed-text",
        input=text
    )
    return np.array(response["embeddings"])

gallup_embeddings = embed(gallup_descriptions)

In [15]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize


def classify_tweet(tweet: str):
    tweet_embedding = embed([tweet])
    similarities: np.ndarray = cosine_similarity(tweet_embedding, gallup_embeddings)[0]
    best_idx = similarities.argmax()

    return gallup_issues[best_idx]

def split_tweet_into_sentences(tweet: str):
    return sent_tokenize(tweet)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jonzd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
def classify_tweet(tweet: str):
    tweet_embedding = embed(tweet)
    similarities: np.ndarray = cosine_similarity(tweet_embedding, gallup_embeddings)[0]
    best_idx = similarities.argmax()

    best_score = similarities[best_idx]
    
    return gallup_issues[best_idx], best_score

In [7]:
### testing on 1 row of df
print(df_sample['text'].iloc[0])
print(classify_tweet(df_sample['text'].iloc[0]))

@kangaroos991 Everything else is in the past. So be it, what you choose to think. But how has Biden saved anything? The most common items are now 5x what they were 4 years ago. He’s just giving trillions away to other nations. Didn’t help Maui at all. Killed Americans, and opened the border???
('Immigration', 0.6502842307153736)


In [ ]:
### testing qwen model
import ollama

response = ollama.generate(
    model='qwen3:8b',
    prompt="Classify this text into Economy, Immigration, Crime or Other: Gas prices are too high",
)

print(response['response'])

In [9]:
### cleaning tweet
import re

def clean_tweet(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [10]:
issues = []

for i, tweet in enumerate(df_sample['text']):
    tweet = clean_tweet(tweet)
    print(f"-----------------Iteration: {i}-----------------")
    issue = classify_tweet(tweet)
    issues.append(issue)

df_sample['issue'] = issues
df_sample.to_csv("../data/issue_mapping_sample.csv", index=False)

-----------------Iteration: 0-----------------
-----------------Iteration: 1-----------------
-----------------Iteration: 2-----------------
-----------------Iteration: 3-----------------
-----------------Iteration: 4-----------------
-----------------Iteration: 5-----------------
-----------------Iteration: 6-----------------
-----------------Iteration: 7-----------------
-----------------Iteration: 8-----------------
-----------------Iteration: 9-----------------
-----------------Iteration: 10-----------------
-----------------Iteration: 11-----------------
-----------------Iteration: 12-----------------
-----------------Iteration: 13-----------------
-----------------Iteration: 14-----------------
-----------------Iteration: 15-----------------
-----------------Iteration: 16-----------------
-----------------Iteration: 17-----------------
-----------------Iteration: 18-----------------
-----------------Iteration: 19-----------------
-----------------Iteration: 20-----------------
--

In [11]:
pd.set_option('display.max_colwidth', None)
print(df_sample[['text', 'issue']])

                                                                                                                                                                                                                                                                                                                        text  \
0                     @kangaroos991 Everything else is in the past. So be it, what you choose to think. But how has Biden saved anything? The most common items are now 5x what they were 4 years ago. He’s just giving trillions away to other nations. Didn’t help Maui at all. Killed Americans, and opened the border???   
1                                                                                                                                                                                                                        Man who manipulates rates, thus inflation to help Biden get re-elected wonders why so many are mad?   
2                                       

In [12]:
df_sample['clean_text'] = df_sample['text'].apply(clean_tweet)

results = df_sample['clean_text'].apply(classify_tweet)
df_sample[['issue_semantic', 'similarity']] = pd.DataFrame(results.tolist(), index=df_sample.index)
avg_similarity = df_sample['similarity'].mean()
print("Average similarity:", df_sample['similarity'].mean())
print("Max similarity:", df_sample['similarity'].max())
print("Min similarity:", df_sample['similarity'].min())

Average similarity: 0.5904008556217067
Max similarity: 0.6990780157526288
Min similarity: 0.46849292986549673


In [13]:
try:
    df_sample.to_csv("../data/sample_issue_classification.csv", index=False)
    print("Saved dataset: ../data/sample_issue_classification.csv")
except Exception as e:
    print("Error while saving:", e)

Saved dataset: ../data/sample_issue_classification.csv


In [17]:
def embed_batch(texts):
    response = ollama.embed(
        model="nomic-embed-text",
        input=texts
    )
    return np.array(response["embeddings"])

new_tweet_embeddings = embed_batch(df['clean_text'].tolist())
similarities = cosine_similarity(new_tweet_embeddings, gallup_embeddings)

best_indices = similarities.argmax(axis=1)
best_scores = similarities.max(axis=1)

results = [(gallup_issues[i], score) for i, score in zip(best_indices, best_scores)]

In [22]:
df['issue_semantic'] = [gallup_issue for gallup_issue, _ in results]
df['similarity'] = [score for _, score in results]

print("Average similarity:", df['similarity'].mean())
print("Max similarity:", df['similarity'].max())
print("Min similarity:", df['similarity'].min())

Average similarity: 0.5800782898096462
Max similarity: 0.8454978882048843
Min similarity: 0.3241013232474955


In [23]:
tweets_per_issue = df.groupby('issue_semantic').size().sort_values(ascending=False)
print(tweets_per_issue)

users_per_issue = df.groupby('issue_semantic')['user'].nunique().sort_values(ascending=False)
print(users_per_issue)

similarity_per_issue = df.groupby('issue_semantic')['similarity'].mean().sort_values(ascending=False)
print(similarity_per_issue)

issue_semantic
Situation in Middle East between Israelis and Palestinians    53374
Democracy in the U.S.                                         47337
Crime                                                         21773
Relations with China                                          19126
Types of Supreme Court justices candidates would pick         17749
Relations with Russia                                         15405
Immigration                                                   12850
Terrorism and national security                                9176
Gun policy                                                     7203
Distribution of income and wealth in the U.S.                  6373
Climate change                                                 6278
The federal budget deficit                                     6014
Foreign affairs                                                5671
Taxes                                                          4067
The economy                      

In [24]:
issue_summary = pd.DataFrame({
    'tweet_count': df.groupby('issue_semantic').size(),
    'unique_users': df.groupby('issue_semantic')['user'].nunique(),
    'avg_similarity': df.groupby('issue_semantic')['similarity'].mean()
})

issue_summary = issue_summary.sort_values(by='tweet_count', ascending=False)

print(issue_summary)

                                                            tweet_count  \
issue_semantic                                                            
Situation in Middle East between Israelis and Palestinians        53374   
Democracy in the U.S.                                             47337   
Crime                                                             21773   
Relations with China                                              19126   
Types of Supreme Court justices candidates would pick             17749   
Relations with Russia                                             15405   
Immigration                                                       12850   
Terrorism and national security                                    9176   
Gun policy                                                         7203   
Distribution of income and wealth in the U.S.                      6373   
Climate change                                                     6278   
The federal budget defici

In [31]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

issue_counts = df['issue_semantic'].value_counts().reset_index()
issue_counts.columns = ['Issue', 'Tweet Count']

fig = px.bar(
    issue_counts.sort_values('Tweet Count'),
    x='Tweet Count',
    y='Issue',
    orientation='h',
    text='Tweet Count',
    title='Tweets per Issue (Semantic Classification)'
)

fig.update_layout(
    yaxis=dict(categoryorder='total ascending'),
    height=600
)

fig.show()

In [33]:
top_issues = issue_counts.sort_values('Tweet Count', ascending=False).head(10)

fig = px.bar(
    top_issues.sort_values('Tweet Count'),
    x='Tweet Count',
    y='Issue',
    orientation='h',
    text='Tweet Count',
    title='Top 10 Issues Discussed on Twitter'
)

fig.show()